# Volatility Dynamics 17 — Conditional variance tuning for Student-t return Markov dynamics

**Status: registered historical exploration; no real-market run yet.** Research plan: [VD 034](../../discussions/Volatility_Dynamics/034_chatgpt_vd17_conditional_variance_preregistration.md).

Seven horizons: **5, 10, 15, 30, 60, 120 and 240 minutes**; 60m primary. This experiment **does not change Student-t return transitions, the two-factor Markov volatility transition matrix or the global VD16 tail shape**. It tunes conditional forecast variance with either current causal state information, M2's independently TRAIN-fitted variance scale, or both. The latter is a **hybrid forecast**, not a pure Markov process. Output scale overlays need not be cross-horizon path-consistent.

The historical TEST segment was already viewed in VD13–16; nothing here is prospective confirmation or a trading signal.

## 0. Environment and frozen protocol

In [ ]:
from pathlib import Path
import os,sys,json,time
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run from project repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_conditional_variance as v17
DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(), 'Set DERIBIT_DATA_ROOT in .env'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(), 'Missing history/price_5m folder'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'17'
OUT.mkdir(exist_ok=True,parents=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
assert v17.PARTICLES==256 and v17.BOOT==1000 and v17.CAL_LIMIT==512
start=time.monotonic()
print(json.dumps(v17.registration(),indent=2,default=str))

## 1. Synthetic and regression tests (no real market-data use)

In [ ]:
import subprocess
p=subprocess.run([sys.executable,'-m','pytest','-q',
  str(ROOT/'tests'/'test_markov_conditional_variance.py'),
  str(ROOT/'tests'/'test_markov_return_horizon_calibration.py'),
  str(ROOT/'tests'/'test_student_t_return_markov.py'),
  str(ROOT/'tests'/'test_markov_lift_distributions.py')],
  cwd=str(ROOT),capture_output=True,text=True)
print(p.stdout[-5000:]); print(p.stderr[-1500:])
assert p.returncode==0, 'Tests must pass before market data evaluation' 

## 2. Native five-minute history, seven horizon targets

Read **Binance spot** close prices only. Features are causal, training-only clock centering; missing bars and split crossing are censored. Historical data ends strictly before 2026-09-19 UTC.

In [ ]:
raw,excluded=v14.load_returns(HIST,ASSETS)
panel=v14.panelize(raw)
del raw
features=v14.make_features(panel)
frame=v14.make_targets(panel,features,v17.HORIZONS)
print('Assets:', sorted(panel.underlying.unique()),'| excluded:',excluded)
print(panel.groupby('split').agg(days=('date','nunique'),rows=('return_5m','size')).to_string())
print('Elapsed',round(time.monotonic()-start,1),'s') 

## 3. Fit conditional variance overlays; historical retrospective score

The native Student-t return process is refitted on TRAIN with original VD15 nu selection using VALID; VD16 base scale and shape are fitted on **early VALID** and never touched thereafter. Conditional variance corrections fit on the same early VALID subsample; late VALID is purged and used only for descriptive selection. Historical TEST scores are exploratory because the period was inspected in prior modules.

In [ ]:
scores,boot,per_asset,variance_fit,params=v17.run_all(frame)
assert len(scores)==len(v17.HORIZONS)*len(v17.MODELS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
print(scores.groupby('horizon_min').agg(models=('model','nunique'),origins=('n_origins','first'),days=('n_days','first')).to_string())
print('Elapsed',round(time.monotonic()-start,1),'s')

## 4. Explainable variance fits, density scores, tail scores and variance QLIKE

A significant NLL gain is **not** enough to promote a model if tail calibration or net-return QLIKE worsens. `AM_T_VAR_ANCHOR` borrows M2's variance *forecast*, not its density; do not interpret it as purely incremental Markov information.

In [ ]:
display(variance_fit.round(6))
display(scores.round(7))
display(boot[(boot.horizon_min==60)&boot.metric.isin(['nll','return_qlike','tail_logloss'])].round(7))
display(per_asset[per_asset.horizon_min==60].round(6))
print('WARNING: historical TEST has been inspected in VD13-16; no prospective confirmation.')

## 5. Numerical QMC convergence on late VALID, not TEST

Check nested 256 and 512 Sobol nodes at 5m, 1h, 4h. The size of QMC-induced NLL changes bounds what small forecasting improvements can mean.

In [ ]:
stability=v17.particle_sensitivity_validation(frame,params['native_kernels'],
  params['markov_transitions'],params['calibrations'],horizons=(5,60,240))
display(stability.round(7))

## 6. Exploratory decision and compact saved outputs

In [ ]:
decision=v17.decide(scores,boot,params)
assert decision['status']=='HISTORICAL_EXPLORATORY'
assert not decision['historical_confirmation_permitted']
print(json.dumps(decision,indent=2))
scores.to_csv(OUT/'17_scores.csv',index=False,float_format='%.10g')
boot.to_csv(OUT/'17_bootstrap.csv',index=False,float_format='%.10g')
per_asset.to_csv(OUT/'17_per_asset.csv',index=False,float_format='%.10g')
variance_fit.to_csv(OUT/'17_variance_fit_grid.csv',index=False,float_format='%.10g')
stability.to_csv(OUT/'17_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'17_decision.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_17','version':v17.VERSION,'registration':v17.registration(),
  'source':str(HIST),'sample_first':str(panel.state_time.min()),'sample_last':str(panel.state_time.max()),
  'assets':sorted(panel.underlying.unique().tolist()),'excluded_assets':excluded,
  'days_by_split':panel.groupby('split').date.nunique().to_dict(),
  'fitted_parameters':params,'decision':decision,
  'warning':'Historical period already inspected; conditional forecasting overlays not a path-consistent Markov process.'}
(OUT/'17_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved 7 compact research artifacts:',OUT)
print('No raw returns, forecast-origin dumps, DuckDB or collector writes.')